In [1]:
import pandas as pd
import numpy as np

file_path = r"E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal\features\PLATE6_T1\A1\3.npz"

with np.load(file_path) as data:
    # Convert the numpy array to a Pandas DataFrame
    df = pd.DataFrame(data['features'])
    
    # Save to CSV
    df.to_csv("site_features_A1_3.csv", index=False)
    print("Saved to site_features_A1_3.csv")

Saved to site_features_A1_3.csv


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import skimage.io
import os
import re

# --- 1. CONFIGURATION ---
loc_dir = r"D:\Thesis\final\project_antibiotics\inputs\locations\PLATE6_T1"
image_dir = r"D:\antibioticscreenTiff"
index_meta_path = r"D:\Thesis\final\project_antibiotics\inputs\metadata\index.csv"
save_base_dir = r"D:\Thesis\final\cropsAntibiotics_extraqual\PLATE6_T1"

plate = "PLATE6_T1"
box_size = 224
half_box = box_size // 2
# Updated to only target C1 and C5
channels_to_crop = ['C1', 'C5']

if not os.path.exists(save_base_dir):
    os.makedirs(save_base_dir)

# --- 2. LOAD & FILTER METADATA ---
print(f"Loading metadata from {index_meta_path}...")
idx_df = pd.read_csv(index_meta_path)

# Filter for PLATE6_T1 and the specific treatment concentrations/controls
# Matches _10, _10_1, _10_2, _50, _50_1, _50_2 and controls
treatment_pattern = r'(_10(_[12])?|_50(_[12])?|no_sgRNA|nosgrna)'

df_plate = idx_df[
    (idx_df['Metadata_Plate'] == plate) & 
    (idx_df['Treatment'].str.contains(treatment_pattern, case=False, na=False, regex=True))
].copy()

unique_wells = df_plate['Metadata_Well'].unique()
print(f"Filtered to {len(unique_wells)} wells. Starting full site processing...")

# --- 3. THE MAIN AUTOMATION LOOP ---
for well_idx, target_well in enumerate(unique_wells):
    # Get ALL sites for this well
    df_well = df_plate[idx_df['Metadata_Well'] == target_well]
    
    print(f"\n--- Processing Well [{well_idx+1}/{len(unique_wells)}]: {target_well} ({len(df_well)} sites) ---")

    for _, site_row in df_well.iterrows():
        target_site = site_row['Metadata_Site']
        target_treatment = site_row['Treatment']
        safe_treatment = str(target_treatment).replace("/", "_").replace(" ", "_")

        # Create subfolder for this specific Well + Site
        well_subfolder_name = f"{plate}_{target_well}_XY{target_site}_{safe_treatment}"
        target_subfolder = os.path.join(save_base_dir, well_subfolder_name)
        
        if not os.path.exists(target_subfolder):
            os.makedirs(target_subfolder)

        # --- 4. LOAD LOCATIONS ---
        loc_file = os.path.join(loc_dir, f"{target_well}-{target_site}-Nuclei.csv")
        if not os.path.exists(loc_file):
            print(f"    Skipping: Location file {target_well}-{target_site} not found.")
            continue

        site_data = pd.read_csv(loc_file)

        # --- 5. LOAD IMAGES (C1 and C5 only) ---
        full_images = {}
        for ch in channels_to_crop:
            if ch in site_row:
                rel_path = str(site_row[ch]).replace('/', os.sep)
                full_path = os.path.join(image_dir, rel_path)
                
                if os.path.exists(full_path):
                    try:
                        img = skimage.io.imread(full_path, plugin='tifffile')
                        # Ensure 2D (take first slice if multi-page)
                        full_images[ch] = img[0] if img.ndim > 2 else img
                    except Exception as e:
                        print(f"    Error reading {ch} for site {target_site}: {e}")

        if not full_images:
            continue

        # --- 6. CROP AND SAVE ---
        crops_saved_count = 0
        for _, cell in site_data.iterrows():
            # Get Cell ID (ObjectNumber) and Coordinates
            cell_id = int(cell.get('ObjectNumber', _))
            cx = int(cell['Nuclei_Location_Center_X'])
            cy = int(cell['Nuclei_Location_Center_Y'])

            for ch_key, ch_full in full_images.items():
                # Define crop boundaries with clamping
                y1, y2 = max(0, cy-half_box), min(ch_full.shape[0], cy+half_box)
                x1, x2 = max(0, cx-half_box), min(ch_full.shape[1], cx+half_box)
                
                crop = ch_full[y1:y2, x1:x2]
                
                # Normalization (0-255) for manual QC visibility
                c_min, c_max = crop.min(), np.percentile(crop, 99.9)
                if c_max > c_min:
                    crop_viz = np.clip((crop - c_min) / (c_max - c_min) * 255, 0, 255).astype(np.uint8)
                else:
                    crop_viz = np.zeros_like(crop, dtype=np.uint8)

                # Filename: Cell_1_C1_X1465_Y1493_TreatmentName.png
                file_name = f"Cell_{cell_id}_{ch_key}_X{cx}_Y{cy}_{safe_treatment}.png"
                save_path = os.path.join(target_subfolder, file_name)
                
                skimage.io.imsave(save_path, crop_viz, check_contrast=False)
                crops_saved_count += 1

        print(f"    Done. Saved {crops_saved_count} crops (C1/C5) for Site {target_site}.")
        
        # Memory Cleanup for next site
        del full_images

print("\nProcessing complete! All sites for selected treatments have been cropped.")

Loading metadata from D:\Thesis\final\project_antibiotics\inputs\metadata\index.csv...


C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\3474426290.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  (idx_df['Treatment'].str.contains(treatment_pattern, case=False, na=False, regex=True))
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\3474426290.py:42: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_well = df_plate[idx_df['Metadata_Well'] == target_well]
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\3474426290.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')


Filtered to 30 wells. Starting full site processing...

--- Processing Well [1/30]: G1 (13 sites) ---


C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\3474426290.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')


    Done. Saved 54 crops (C1/C5) for Site 6.


...

    Done. Saved 20 crops (C1/C5) for Site 13.


C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\3474426290.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\3474426290.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')


    Done. Saved 24 crops (C1/C5) for Site 14.


C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\3474426290.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\3474426290.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')


    Done. Saved 40 crops (C1/C5) for Site 15.

Processing complete! All sites for selected treatments have been cropped.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import skimage.io
import os
import re

# --- 1. CONFIGURATION ---
loc_dir = r"D:\Thesis\final\project_antibiotics\inputs\locations\PLATE7_T1"
image_dir = r"D:\antibioticscreenTiff"
index_meta_path = r"D:\Thesis\final\project_antibiotics\inputs\metadata\index.csv"
save_base_dir = r"D:\Thesis\final\cropsAntibiotics_extraqual\PLATE7_T1"

plate = "PLATE7_T1"
box_size = 224
half_box = box_size // 2
# Updated to only target C1 and C5
channels_to_crop = ['C1', 'C5']

if not os.path.exists(save_base_dir):
    os.makedirs(save_base_dir)

# --- 2. LOAD & FILTER METADATA ---
print(f"Loading metadata from {index_meta_path}...")
idx_df = pd.read_csv(index_meta_path)

# Filter for PLATE6_T1 and the specific treatment concentrations/controls
# Matches _10, _10_1, _10_2, _50, _50_1, _50_2 and controls
treatment_pattern = r'(_10(_[12])?|_50(_[12])?|no_sgRNA|nosgrna)'

df_plate = idx_df[
    (idx_df['Metadata_Plate'] == plate) & 
    (idx_df['Treatment'].str.contains(treatment_pattern, case=False, na=False, regex=True))
].copy()

unique_wells = df_plate['Metadata_Well'].unique()
print(f"Filtered to {len(unique_wells)} wells. Starting full site processing...")

# --- 3. THE MAIN AUTOMATION LOOP ---
for well_idx, target_well in enumerate(unique_wells):
    # Get ALL sites for this well
    df_well = df_plate[idx_df['Metadata_Well'] == target_well]
    
    print(f"\n--- Processing Well [{well_idx+1}/{len(unique_wells)}]: {target_well} ({len(df_well)} sites) ---")

    for _, site_row in df_well.iterrows():
        target_site = site_row['Metadata_Site']
        target_treatment = site_row['Treatment']
        safe_treatment = str(target_treatment).replace("/", "_").replace(" ", "_")

        # Create subfolder for this specific Well + Site
        well_subfolder_name = f"{plate}_{target_well}_XY{target_site}_{safe_treatment}"
        target_subfolder = os.path.join(save_base_dir, well_subfolder_name)
        
        if not os.path.exists(target_subfolder):
            os.makedirs(target_subfolder)

        # --- 4. LOAD LOCATIONS ---
        loc_file = os.path.join(loc_dir, f"{target_well}-{target_site}-Nuclei.csv")
        if not os.path.exists(loc_file):
            print(f"    Skipping: Location file {target_well}-{target_site} not found.")
            continue

        site_data = pd.read_csv(loc_file)

        # --- 5. LOAD IMAGES (C1 and C5 only) ---
        full_images = {}
        for ch in channels_to_crop:
            if ch in site_row:
                rel_path = str(site_row[ch]).replace('/', os.sep)
                full_path = os.path.join(image_dir, rel_path)
                
                if os.path.exists(full_path):
                    try:
                        img = skimage.io.imread(full_path, plugin='tifffile')
                        # Ensure 2D (take first slice if multi-page)
                        full_images[ch] = img[0] if img.ndim > 2 else img
                    except Exception as e:
                        print(f"    Error reading {ch} for site {target_site}: {e}")

        if not full_images:
            continue

        # --- 6. CROP AND SAVE ---
        crops_saved_count = 0
        for _, cell in site_data.iterrows():
            # Get Cell ID (ObjectNumber) and Coordinates
            cell_id = int(cell.get('ObjectNumber', _))
            cx = int(cell['Nuclei_Location_Center_X'])
            cy = int(cell['Nuclei_Location_Center_Y'])

            for ch_key, ch_full in full_images.items():
                # Define crop boundaries with clamping
                y1, y2 = max(0, cy-half_box), min(ch_full.shape[0], cy+half_box)
                x1, x2 = max(0, cx-half_box), min(ch_full.shape[1], cx+half_box)
                
                crop = ch_full[y1:y2, x1:x2]
                
                # Normalization (0-255) for manual QC visibility
                c_min, c_max = crop.min(), np.percentile(crop, 99.9)
                if c_max > c_min:
                    crop_viz = np.clip((crop - c_min) / (c_max - c_min) * 255, 0, 255).astype(np.uint8)
                else:
                    crop_viz = np.zeros_like(crop, dtype=np.uint8)

                # Filename: Cell_1_C1_X1465_Y1493_TreatmentName.png
                file_name = f"Cell_{cell_id}_{ch_key}_X{cx}_Y{cy}_{safe_treatment}.png"
                save_path = os.path.join(target_subfolder, file_name)
                
                skimage.io.imsave(save_path, crop_viz, check_contrast=False)
                crops_saved_count += 1

        print(f"    Done. Saved {crops_saved_count} crops (C1/C5) for Site {target_site}.")
        
        # Memory Cleanup for next site
        del full_images

print("\nProcessing complete! All sites for selected treatments have been cropped.")

Loading metadata from D:\Thesis\final\project_antibiotics\inputs\metadata\index.csv...
Filtered to 49 wells. Starting full site processing...

--- Processing Well [1/49]: C4 (20 sites) ---


C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:33: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  (idx_df['Treatment'].str.contains(treatment_pattern, case=False, na=False, regex=True))
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:42: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_well = df_plate[idx_df['Metadata_Well'] == target_well]
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:75: FutureWarning: 

    Done. Saved 12 crops (C1/C5) for Site 6.


C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')


    Done. Saved 12 crops (C1/C5) for Site 7.


...

    Done. Saved 10 crops (C1/C5) for Site 15.


C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')


    Done. Saved 50 crops (C1/C5) for Site 16.


C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')
C:\Users\arnou\AppData\Local\Temp\ipykernel_16000\4268262692.py:75: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage.io.imread(full_path, plugin='tifffile')


    Done. Saved 88 crops (C1/C5) for Site 17.

Processing complete! All sites for selected treatments have been cropped.
